In [1]:
import pickle
import numpy as np
import pandas as pd
import pywt
import torch
import torch.nn as nn

# =========================
# CONFIG
# =========================
MODEL_PKL = r"C:\Users\akvsk\Downloads\Khyathi\Project\pca_dwt_lstm_bundle.pkl"
TEST_CSV  = r"C:\Users\akvsk\Downloads\Khyathi\Project\sample_test_station.csv"
LOOKBACK = 8

# =========================
# DWT FUNCTION (same as training)
# =========================
def dwt_denoise(x, wavelet="db4", level=2):
    x = np.ascontiguousarray(np.array(x, dtype=np.float64, copy=True))
    coeffs = pywt.wavedec(x, wavelet, level=level, mode="periodization")
    sigma = (np.median(np.abs(coeffs[-1])) / 0.6745) if len(coeffs[-1]) else 0.0
    uth = sigma * np.sqrt(2 * np.log(len(x))) if sigma > 0 else 0.0
    new = [coeffs[0]] + [pywt.threshold(c, value=uth, mode="soft") for c in coeffs[1:]]
    rec = pywt.waverec(new, wavelet, mode="periodization")
    return rec[:len(x)]

# =========================
# LSTM MODEL DEFINITION
# =========================
class LSTMReg(nn.Module):
    def __init__(self, input_dim, hidden=32):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden, batch_first=True)
        self.head = nn.Sequential(
            nn.Linear(hidden, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(out[:, -1, :])

# =========================
# LOAD MODEL BUNDLE
# =========================
with open(MODEL_PKL, "rb") as f:
    bundle = pickle.load(f)

scaler = bundle["scaler"]
pca = bundle["pca"]
model_state = bundle["model_state_dict"]
meta = bundle["pipeline_meta"]
params = bundle["model_params"]

exo_cols = meta["exo_cols"]
feat_cols = meta["feat_cols"]

# =========================
# LOAD TEST DATA
# =========================
df = pd.read_csv(TEST_CSV)
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date")

# Need at least LOOKBACK rows
assert len(df) >= LOOKBACK, "Not enough historical data for prediction"

# =========================
# APPLY DWT ON GWL
# =========================
df["gwl_denoised"] = dwt_denoise(df["gwl"].values)

# =========================
# SCALER + PCA
# =========================
X_exo = scaler.transform(df[exo_cols])
X_pca = pca.transform(X_exo)

for i in range(X_pca.shape[1]):
    df[f"PC{i+1}"] = X_pca[:, i]

# =========================
# BUILD LAST SEQUENCE
# =========================
X_seq = df[feat_cols].values[-LOOKBACK:]
X_seq = torch.tensor(X_seq, dtype=torch.float32).unsqueeze(0)

# =========================
# LOAD MODEL
# =========================
device = "cuda" if torch.cuda.is_available() else "cpu"

model = LSTMReg(
    input_dim=params["input_dim"],
    hidden=params["hidden"]
).to(device)

model.load_state_dict(model_state)
model.eval()

# =========================
# PREDICTION
# =========================
with torch.no_grad():
    pred = model(X_seq.to(device)).cpu().numpy()[0, 0]

print("\n==============================")
print("Predicted Groundwater Level")
print(f"Next Quarter GWL: {pred:.2f} meters")
print("==============================")



Predicted Groundwater Level
Next Quarter GWL: 3.84 meters


c:\Users\akvsk\Downloads\Khyathi\Project\venv\Lib\site-packages\pywt\_multilevel.py:43: UserWarning: Level value of 2 is too high: all coefficients will experience boundary effects.
  warnings.warn(
